In [ ]:
#use pandas import to read csv files
import pandas as pd
#read the csv file provided
data = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vQonF2po8I6w88YhDM8S1WhAESqf8JSyTEbx1fA5mXx9J1eona_QRD8GVehmlA2eNnuYILGKUMDpqQN/pub?gid=36274544&single=true&output=csv')

#split data into training and testing sets
training = data.head(20)
testing = data.tail(10)

#seperate Target from training set to compute probabilities
trainingTargets = training['Target']

#initialize variables to store counts of different data
totalCount = 0
spamCount = 0
hamCount = 0

#loop through and count the targets to compute probabilities
for target in trainingTargets:
  totalCount += 1
  if target == 'spam':
    spamCount += 1
  elif target == 'ham':
    hamCount += 1
#calculate probabilities
spamProbability = spamCount / totalCount
hamProbability = hamCount / totalCount

#create empty dictionaries to append words to depending on whether or not a word is spam or ham
spamDictionary = {}
hamDictionary = {}
#loop through the training data by row
for index, row in training.iterrows():
  #depending on what the target is split the word and add to either spam or ham dictionary
  if row['Target'] == "spam":
    words = row['data'].split()
    for word in words:
      spamDictionary[word] = spamDictionary.get(word, 0) + 1
  elif row['Target'] == "ham":
    words = row['data'].split()
    for word in words:
      hamDictionary[word] = hamDictionary.get(word, 0) + 1

#find total number of words for spam and ham
spamWords = sum(spamDictionary.values())
hamWords = sum(hamDictionary.values())

#find unique words for laplace smoothing
spamUnique = set(spamDictionary.keys())
hamUnique = set(hamDictionary.keys())
uniqueWords = spamUnique.union(hamUnique)

#set total unique to length of unique words
totalUnique = len(uniqueWords)

#calculate conditional probability for spam
spamConditional = {}
for word in uniqueWords:
  spamConditional[word] = (spamDictionary.get(word, 0) + 1) / (spamWords + totalUnique)
#calculate conditional probability for ham
hamConditional = {}
for word in uniqueWords:
  hamConditional[word] = (hamDictionary.get(word, 0) + 1) / (hamWords + totalUnique)

#function for calculating posterior spam and ham probabilities
def spamOrHam(sentence):
  #print sentence
  print(sentence)
  #split sentence
  words = sentence.split()
  #Initialize probabilities as their prior values
  spamProb = spamProbability
  hamProb = hamProbability
  #loop through words
  for word in words:
    #multiply current posterior probability by the current words conditional probability
    spamProb*= spamConditional.get(word, 1)
    hamProb *= hamConditional.get(word, 1)
  #print the posterior probabilities for spam/ham
  print("Spam Probability: ", spamProb)
  print("Ham Probability: ", hamProb)
  #boolean to store whether or not the result is spam
  spamBool = spamProb > hamProb
  #if spam return spam otherwise return ham
  if spamBool:
    return 'spam'
  else:
    return 'ham'
#create variables to store correct results and total sentences to calculate accuracy
correctResults = 0
totalSentences = 0
#loop through all testing rows
for index, row in testing.iterrows():
  #increment total sentences
  totalSentences += 1
  #initialize spam flag as false
  spam = False
  #if the message is spam set spam flag to true
  if(row['Target'] == "spam"):
    spam = True
  #run the sentence through posterior probability function
  result = spamOrHam(row['data'])
  #print the result
  print("Result: ", result)
  #check if the result is correct if so print correct and increment correctResults
  if(result == 'spam' and spam):
    print("Correct")
    print()
    correctResults += 1
  elif(result == 'ham' and not spam):
    print("Correct")
    print()
    correctResults += 1
  #if result is incorrect then print incorrect
  else:
    print("Incorrect")
    print()
#print the accuracy of using the naive bayes algorithm on the testing set
print("Accuracy: ", correctResults / totalSentences)






Tell where you reached
Spam Probability:  0.006617647058823529
Ham Probability:  0.01284046692607004
Result:  ham
Correct

Your gonna have to pick up a burger for yourself on your way home
Spam Probability:  3.501902685098722e-20
Ham Probability:  4.3181309045870886e-20
Result:  ham
Correct

As a valued customer I am pleased to advise you that for your recent review you are awarded a Bonus Prize
Spam Probability:  3.022520865276727e-32
Ham Probability:  4.855590735349672e-32
Result:  ham
Incorrect

Urgent you are awarded a complimentary trip to EuroDisinc To claim text immediately
Spam Probability:  4.626513718827569e-15
Ham Probability:  5.347654277195111e-16
Result:  spam
Correct

Finished class where are you
Spam Probability:  3.8927335640138406e-05
Ham Probability:  9.992581265424156e-05
Result:  ham
Correct

where are you how did you perform
Spam Probability:  4.9520832027450646e-12
Ham Probability:  3.1788649561901397e-10
Result:  ham
Correct

you can call me now
Spam Probability